### 02 Data Preprocessing
 
**Input:** `data/train_eda.csv`, `data/test.csv`  
**Output:** `data/train_preprocessed.csv`, `data/test_preprocessed.csv`


#### Objectives

This notebook handles all data cleaning and encoding decisions before feature engineering and modeling. Specifically:

1. **Drop irrelevant columns**: identifiers and raw timestamps already extracted in Phase 1
2. **Handle missing promo values**: ~30% of rows have no promotion data
3. **Encode categorical variables**: convert string categories to integers for LightGBM
4. **Apply same transformations to test set**: ensuring train/test consistency
5. **Save preprocessed datasets**: for Phase 3 Feature Engineering


#### Design Decisions

| Decision | Choice | Reason |
|---|---|---|
| Drop `session_id` | Yes | it is a UUID identifier with no predictive signal |
| Drop `session_start`, `session_end`, `last_activity` | Yes | Already extracted into `session_duration_s`, `inactivity_gap_s`, `hour_of_day`, `day_of_week` in Phase 1 |
| Drop `min_spend_required` | Yes | 0.93 Pearson correlation with `discount_value` is very high, so this is a near-duplicate, which adds multicollinearity |
| Missing promo values | Add `has_promo` flag + fill with 0/`NONE` | Missingness is structural (session had no promo shown), not random, and `has_promo=0` captures this signal explicitly |
| Categorical encoding | Label encoding | LightGBM handles integer-encoded categoricals natively and efficiently; one-hot encoding is unnecessary and adds dimensionality |

#### 1. Imports & Setup

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

BASE_DIR  = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_DIR  = os.path.join(BASE_DIR, 'data')

TRAIN_IN  = os.path.join(DATA_DIR, 'train_eda.csv')
TEST_IN   = os.path.join(DATA_DIR, 'test.csv')
TRAIN_OUT = os.path.join(DATA_DIR, 'train_preprocessed.csv')
TEST_OUT  = os.path.join(DATA_DIR, 'test_preprocessed.csv')

print(f'Train input : {TRAIN_IN}')
print(f'Test input  : {TEST_IN}')
print(f'Train output: {TRAIN_OUT}')
print(f'Test output : {TEST_OUT}')

Train input : c:\Users\User\OneDrive - Lebanese American University\University\SPRING 2026\COE 546 Machine Learning\Term Project\user_prediction_app\data\train_eda.csv
Test input  : c:\Users\User\OneDrive - Lebanese American University\University\SPRING 2026\COE 546 Machine Learning\Term Project\user_prediction_app\data\test.csv
Train output: c:\Users\User\OneDrive - Lebanese American University\University\SPRING 2026\COE 546 Machine Learning\Term Project\user_prediction_app\data\train_preprocessed.csv
Test output : c:\Users\User\OneDrive - Lebanese American University\University\SPRING 2026\COE 546 Machine Learning\Term Project\user_prediction_app\data\test_preprocessed.csv


#### 2. Load Data

In [2]:
train = pd.read_csv(TRAIN_IN)
test  = pd.read_csv(TEST_IN)

print(f'Train shape: {train.shape}')
print(f'Test shape : {test.shape}')
train.head(3)

Train shape: (297236, 23)
Test shape : (99639, 17)


,id,session_id,session_start,session_end,last_activity,timezone,action_type,promos_declined,customer_type,items_in_cart,...,min_spend_required,promos_shown,screen_size,promo_response,order_placed,session_duration_s,inactivity_gap_s,hour_of_day,day_of_week,is_weekend
0,0,082f3074-39b5-448d-b696-1e99de783de7,2017-03-07 00:17:41.049000+00:00,2017-03-07 00:47:41.049000+00:00,2017-03-07 00:40:04.857000+00:00,420,PAGE_LOAD,0,NC,0,...,50.0,1.0,1048576.0,IGNORED,0,1800.0,456.192,0,1,0
1,2,70b9ed34-4de3-47c4-ae00-2a0f4cd4579f,2017-03-03 07:17:25.925000+00:00,2017-03-03 07:47:25.925000+00:00,2017-03-03 07:42:06.853000+00:00,480,PAGE_LOAD,0,NC,0,...,950.0,1.0,1048576.0,IGNORED,0,1800.0,319.072,7,4,0
2,3,5838186d-164f-4f51-bd35-c82a293d5e14,2017-03-06 14:31:23.833000+00:00,2017-03-06 15:01:23.833000+00:00,2017-03-06 14:51:43.569000+00:00,480,PAGE_LOAD,0,NC,0,...,50.0,1.0,1048576.0,IGNORED,0,1800.0,580.264,14,0,0


#### 3. Rename Test Columns & Derive Timestamp Features

The test set has the original anonymized column names (`f2`–`f17`) and does **not** yet have the derived features (`session_duration_s`, `inactivity_gap_s`, etc.) that were computed in Phase 1 for the train set. We apply the same renaming and timestamp derivation here.

In [ ]:
#Rename test columns to match train
test = test.rename(columns={
    'f2' : 'session_id',
    'f3' : 'session_start',
    'f4' : 'session_end',
    'f5' : 'last_activity',
    'f6' : 'timezone',
    'f7' : 'action_type',
    'f8' : 'promos_declined',
    'f9' : 'customer_type',
    'f10': 'items_in_cart',
    'f11': 'cart_value',
    'f12': 'promo_type',
    'f13': 'discount_value',
    'f14': 'min_spend_required',
    'f15': 'promos_shown',
    'f16': 'screen_size',
    'f17': 'promo_response',
})

#Derive timestamp features for test (same logic as Phase 1)
test['session_start']  = pd.to_datetime(test['session_start'],  utc=True)
test['session_end']    = pd.to_datetime(test['session_end'],    utc=True)
test['last_activity']  = pd.to_datetime(test['last_activity'],  utc=True)

test['session_duration_s'] = (test['session_end'] - test['session_start']).dt.total_seconds()
test['inactivity_gap_s']   = (test['session_end'] - test['last_activity']).dt.total_seconds()
test['hour_of_day']        = test['session_start'].dt.hour
test['day_of_week']        = test['session_start'].dt.dayofweek
test['is_weekend']         = test['day_of_week'].isin([5, 6]).astype(int)

print('Test columns after rename + derivation:')
print(test.columns.tolist())

Test columns after rename + derivation:
['id', 'session_id', 'session_start', 'session_end', 'last_activity', 'timezone', 'action_type', 'promos_declined', 'customer_type', 'items_in_cart', 'cart_value', 'promo_type', 'discount_value', 'min_spend_required', 'promos_shown', 'screen_size', 'promo_response', 'session_duration_s', 'inactivity_gap_s', 'hour_of_day', 'day_of_week', 'is_weekend']


#### 4. Drop Irrelevant Columns

**Dropped:**
- `session_id`
- `session_start`, `session_end`, `last_activity`
- `min_spend_required`

In [4]:
COLS_TO_DROP = [
    'session_id',
    'session_start',
    'session_end',
    'last_activity',
    'min_spend_required',
]

train = train.drop(columns=COLS_TO_DROP)
test  = test.drop(columns=COLS_TO_DROP)

print(f'Train shape after dropping: {train.shape}')
print(f'Test shape after dropping : {test.shape}')
print(f'\nRemaining columns: {train.columns.tolist()}')

Train shape after dropping: (297236, 18)
Test shape after dropping : (99639, 17)

Remaining columns: ['id', 'timezone', 'action_type', 'promos_declined', 'customer_type', 'items_in_cart', 'cart_value', 'promo_type', 'discount_value', 'promos_shown', 'screen_size', 'promo_response', 'order_placed', 'session_duration_s', 'inactivity_gap_s', 'hour_of_day', 'day_of_week', 'is_weekend']


#### 5. Handle Missing Promo Values

The following 5 columns have ~30% missing values:
- `promo_type`, `discount_value`, `promos_shown`, `promo_response` (and we already dropped `min_spend_required`)

**Why they are missing:** These rows represent sessions where no promotion was shown to the user. The missingness is **structural** (tied to a real-world state), not random data corruption.

**Strategy:**
1. Add a binary `has_promo` flag (1 = promo was shown, 0 = no promo). This explicitly encodes the signal
2. Fill numeric NaNs (`discount_value`, `promos_shown`) with `0`
3. Fill categorical NaNs (`promo_type`, `promo_response`) with `'NONE'`

In [ ]:
#Add has_promo flag BEFORE filling NaNs
# We use promo_type as the indicator. it's NaN for all no-promo sessions
train['has_promo'] = train['promo_type'].notna().astype(int)
test['has_promo']  = test['promo_type'].notna().astype(int)

print('has_promo distribution (train):')
print(train['has_promo'].value_counts())
print(f'\nPromo sessions   : {train["has_promo"].sum():,} ({train["has_promo"].mean()*100:.1f}%)')
print(f'No-promo sessions: {(train["has_promo"]==0).sum():,} ({(train["has_promo"]==0).mean()*100:.1f}%)')

has_promo distribution (train):
has_promo
1    206718
0     90518
Name: count, dtype: int64

Promo sessions   : 206,718 (69.5%)
No-promo sessions: 90,518 (30.5%)


In [ ]:
#Fill numeric promo NaNs with 0
NUMERIC_PROMO_COLS = ['discount_value', 'promos_shown']
for col in NUMERIC_PROMO_COLS:
    train[col] = train[col].fillna(0)
    test[col]  = test[col].fillna(0)

#Fill categorical promo NaNs with 'NONE'
CAT_PROMO_COLS = ['promo_type', 'promo_response']
for col in CAT_PROMO_COLS:
    train[col] = train[col].fillna('NONE')
    test[col]  = test[col].fillna('NONE')

#Verify no remaining NaNs
missing_train = train.isnull().sum().sum()
missing_test  = test.isnull().sum().sum()
print(f'Remaining NaNs in train: {missing_train}')
print(f'Remaining NaNs in test : {missing_test}')

if missing_train == 0 and missing_test == 0:
    print('\nNo missing values remaining.')
else:
    print('\nSome NaNs remain. investigate before proceeding.')
    print(train.isnull().sum()[train.isnull().sum() > 0])

Remaining NaNs in train: 0
Remaining NaNs in test : 0

✅ No missing values remaining.


#### 6. Encode Categorical Variables

The 4 categorical string columns are label-encoded (converted to integers).

**Why label encoding over one-hot encoding:**  
LightGBM supports categorical features natively as integers. It builds optimal splits across all categories without requiring dummy variables. One-hot encoding would unnecessarily increase dimensionality and is less efficient with tree-based models.

**Important:** We fit the encoder on the **combined** unique values from both train and test to ensure all categories are handled. This prevents unseen-label errors at inference time.

In [ ]:
CATEGORICAL_COLS = ['action_type', 'customer_type', 'promo_type', 'promo_response']

label_encoders = {}  # Store encoders in case we need to inverse-transform later

for col in CATEGORICAL_COLS:
    le = LabelEncoder()
    
    # Fit on union of train + test values to handle all categories
    combined_values = pd.concat([train[col], test[col]], ignore_index=True)
    le.fit(combined_values)
    
    # Transform both sets
    train[col] = le.transform(train[col])
    test[col]  = le.transform(test[col])
    
    label_encoders[col] = le
    print(f'{col:20s} → classes: {list(le.classes_)}')

print('\nAll categoricals encoded.')

action_type          → classes: ['CART_CHANGE', 'PAGE_LOAD']
customer_type        → classes: ['NC', 'OC']
promo_type           → classes: ['C', 'F', 'NONE', 'P', 'S']
promo_response       → classes: ['ACCEPTED', 'DECLINED', 'IGNORED', 'NONE']

✅ All categoricals encoded.


#### 7. Final Verification

Before saving, we verify:
- Shape and column alignment between train and test
- All dtypes are numeric
- No remaining NaNs

In [ ]:
# Columns in train (excluding target) vs test
train_cols = set(train.columns) - {'order_placed'}
test_cols  = set(test.columns)

only_in_train = train_cols - test_cols
only_in_test  = test_cols  - train_cols

print(f'Train shape : {train.shape}')
print(f'Test shape  : {test.shape}')
print(f'\nColumns only in train (excl. target): {only_in_train}')
print(f'Columns only in test                : {only_in_test}')

print('\n── Train dtypes ──')
print(train.dtypes)

print('\n── Sample rows ──')
train.head(3)

Train shape : (297236, 19)
Test shape  : (99639, 18)

Columns only in train (excl. target): set()
Columns only in test                : set()

── Train dtypes ──
id                      int64
timezone                int64
action_type             int32
promos_declined         int64
customer_type           int32
items_in_cart           int64
cart_value            float64
promo_type              int32
discount_value        float64
promos_shown          float64
screen_size           float64
promo_response          int32
order_placed            int64
session_duration_s    float64
inactivity_gap_s      float64
hour_of_day             int64
day_of_week             int64
is_weekend              int64
has_promo               int32
dtype: object

── Sample rows ──


,id,timezone,action_type,promos_declined,customer_type,items_in_cart,cart_value,promo_type,discount_value,promos_shown,screen_size,promo_response,order_placed,session_duration_s,inactivity_gap_s,hour_of_day,day_of_week,is_weekend,has_promo
0,0,420,1,0,0,0,0.0,1,10.0,1.0,1048576.0,2,0,1800.0,456.192,0,1,0,1
1,2,480,1,0,0,0,0.0,3,75.0,1.0,1048576.0,2,0,1800.0,319.072,7,4,0,1
2,3,480,1,0,0,0,0.0,3,10.0,1.0,1048576.0,2,0,1800.0,580.264,14,0,0,1


In [ ]:
# Summary statistics
print('Train describe:')
train.describe().round(2)

Train describe:


,id,timezone,action_type,promos_declined,customer_type,items_in_cart,cart_value,promo_type,discount_value,promos_shown,screen_size,promo_response,order_placed,session_duration_s,inactivity_gap_s,hour_of_day,day_of_week,is_weekend,has_promo
count,297236.00,297236.00,297236.00,297236.00,297236.00,297236.00,297236.00,297236.00,297236.00,297236.00,297236.00,297236.00,297236.00,297236.00,297236.00,297236.00,297236.00,297236.00,297236.00
mean,198386.13,319.98,0.98,0.00,0.57,0.79,101.89,1.49,10.51,0.94,2447610.38,2.23,0.03,2075.49,1363.99,11.35,3.43,0.43,0.70
std,114587.65,119.13,0.14,0.07,0.49,3.98,729.79,1.01,16.46,2.97,1338905.34,0.61,0.17,998.22,530.89,7.45,2.34,0.50,0.46
min,0.00,-780.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1800.00,1.32,0.00,0.00,0.00,0.00
25%,99248.00,300.00,1.00,0.00,0.00,0.00,0.00,1.00,0.00,0.00,1483776.00,2.00,0.00,1800.00,981.81,3.00,0.00,0.00,0.00
50%,198351.50,300.00,1.00,0.00,1.00,0.00,0.00,1.00,10.00,1.00,2529280.00,2.00,0.00,1800.29,1685.12,13.00,4.00,0.00,1.00
75%,297485.25,420.00,1.00,0.00,1.00,1.00,24.99,2.00,10.00,1.00,3145728.00,3.00,0.00,1985.23,1775.42,18.00,5.00,1.00,1.00
max,396873.00,720.00,1.00,14.00,1.00,288.00,67663.81,4.00,100.00,479.00,37748736.00,3.00,1.00,242637.85,5903.10,23.00,6.00,1.00,1.00


#### 8. Save Preprocessed Data

Saving both preprocessed datasets to the `data/` folder for use in Phase 3 (Feature Engineering).

In [ ]:
train.to_csv(TRAIN_OUT, index=False)
test.to_csv(TEST_OUT,   index=False)

print(f'Saved train_preprocessed.csv → {train.shape}')
print(f'Saved test_preprocessed.csv  → {test.shape}')
print(f'\nFinal feature columns ({len(train.columns)-2} features + id + target):')
feature_cols = [c for c in train.columns if c not in ['id', 'order_placed']]
for i, col in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {col}')

✅ Saved train_preprocessed.csv → (297236, 19)
✅ Saved test_preprocessed.csv  → (99639, 18)

Final feature columns (17 features + id + target):
   1. timezone
   2. action_type
   3. promos_declined
   4. customer_type
   5. items_in_cart
   6. cart_value
   7. promo_type
   8. discount_value
   9. promos_shown
  10. screen_size
  11. promo_response
  12. session_duration_s
  13. inactivity_gap_s
  14. hour_of_day
  15. day_of_week
  16. is_weekend
  17. has_promo


#### 9. Preprocessing Summary

| Step | Action | Train Shape | Test Shape |
|---|---|---|---|
| Loaded | Raw EDA output / test.csv | (297236, 23) | (99639, 17) |
| Renamed test + derived features | Applied same Phase 1 logic to test | — | (99639, 23) |
| Dropped 5 columns | session_id, timestamps ×3, min_spend_required | (297236, 18) | (99639, 18) |
| Added has_promo flag | Binary indicator for promo presence | +1 col | +1 col |
| Filled promo NaNs | 0 for numerics, 'NONE' for categoricals | 0 NaNs | 0 NaNs |
| Label encoded | action_type, customer_type, promo_type, promo_response | all numeric | all numeric |

**Hand-off:** `data/train_preprocessed.csv` and `data/test_preprocessed.csv` are ready for Phase 3 — Feature Engineering (`03_feature_engineering.ipynb`).